In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

# 1. Muat Data
df = pd.read_csv('data.csv')
df = df.dropna()

texts = df['text'].astype(str).values
labels = df['label'].values

# 2. Konfigurasi Parameter
vocab_size = 5000
max_length = 50
embedding_dim = 32

# 3. Pra-pemrosesan Teks
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)
padded = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')

# 4. Pembagian Data (80% Latih, 20% Uji)
X_train, X_test, y_train, y_test = train_test_split(padded, labels, test_size=0.2, random_state=42)

# 5. Arsitektur Model LSTM
model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_length),
    LSTM(64),
    Dropout(0.5), # Regularisasi untuk menekan overfitting
    Dense(1, activation='sigmoid') # Output biner (0 atau 1)
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# 6. Pelatihan
model.fit(X_train, y_train, epochs=10, validation_split=0.1, verbose=1)

# 7. Evaluasi
y_pred_probs = model.predict(X_test)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()

print(f"Akurasi: {accuracy_score(y_test, y_pred):.4f}")
print("Laporan Klasifikasi:\n", classification_report(y_test, y_pred))

c:\Users\LOQ\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - accuracy: 0.4977 - loss: 0.6938 - val_accuracy: 0.5151 - val_loss: 0.6929
Epoch 2/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.4916 - loss: 0.6941 - val_accuracy: 0.4849 - val_loss: 0.6932
Epoch 3/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.8721 - loss: 0.2426 - val_accuracy: 0.9880 - val_loss: 0.0577
Epoch 4/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.9903 - loss: 0.0522 - val_accuracy: 0.9880 - val_loss: 0.0570
Epoch 5/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.9909 - loss: 0.0473 - val_accuracy: 0.9910 - val_loss: 0.0452
Epoch 6/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.9913 - loss: 0.0463 - val_accuracy: 0.9910 - val_loss: 0.0452
Epoch 7/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.9926 - loss: 0.0412 - val_accuracy: 0.9910 - val_loss: 0.0452
Epoch 8/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.9950 - loss: 0.0315 - val_accuracy: 0.9910 - v

In [3]:
import pickle
# Skrip pelatihan lokal
model.save('model_lstm.keras')

# Simpan Tokenizer
with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)